# Tension Board 2: Hold Difficulty Analysis

We continue on with our hold analysis, except we will solely be interested in computing the difficulty of each hold.

Recall some of the following findings.

- TB2 Mirror has has `layout_id` 10, and has two sets: wood and plastic. These have `set_id` 12 and 13 respectively. 
- the `frame` feature of a climb determines the climb: it looks something like `p3r4p29r2p59r1p65r2p75r3p89r2p157r4p158r4`. A substring `pXrY` tells us the placement (`placement_id=X`) and the role (whether it is a start, finish, foot, or middle hold) comes from the `placement_role_id=Y`. The role will also tell us which color to use if we plot our climb against the board.
- the `holes` table will tell us which `placement_id` goes where on the (x,y) coordinate system. It also tells us the ID of its mirror image, which let's us unravel the `placement_id` of its mirror image.

## Output

The final products are hold-level difficulty scores saved to CSV files. These scores encode, for each placement, the average difficulty of climbs that use that hold. The scores are computed per-angle, per-role, and also aggregated. A Bayesian smoothing step shrinks noisy estimates for rarely-used holds toward the global mean, and mirror averaging stabilizes scores across symmetric left-right hold pairs.

## Notebook Structure

1. [Setup and Imports](#setup-and-imports)
2. [Hold Usage DataFrame](#hold-usage-dataframe)
3. [Difficulty Score](#difficulty-score)
4. [Visualization](#visualization)
5. [Conclusion](#conclusion)

# Setup and Imports

In [ ]:
"""
==================================
Setup and Imports
==================================
"""


# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.patches as mpatches

import sqlite3

import os

import re
from collections import defaultdict

from PIL import Image

# Set some display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set style
palette=['steelblue', 'coral', 'seagreen']  #(for multi-bar graphs)

# Set board image for some visual analysis
board_img = Image.open('../images/tb2_board_12x12_composite.png')

# Connect to the database
DB_PATH="../data/tb2.db"
conn = sqlite3.connect(DB_PATH)

In [ ]:
"""
==================================
Query our data from the DB
==================================

This time we restrict to where `layout_id=10` for the TB2 Mirror.
"""

# Query climbs data
climbs_query = """
SELECT
    c.uuid,
    c.name AS climb_name,
    c.setter_username,
    c.layout_id AS layout_id,
    c.description,
    c.is_nomatch,
    c.is_listed,
    l.name AS layout_name,
    p.name AS board_name,
    c.frames,
    cs.angle,
    cs.display_difficulty,
    dg.boulder_name AS boulder_grade,
    cs.ascensionist_count,
    cs.quality_average,
    cs.fa_at
    
FROM climbs c
JOIN layouts l ON c.layout_id = l.id
JOIN products p ON l.product_id = p.id
JOIN climb_stats cs ON c.uuid = cs.climb_uuid
JOIN difficulty_grades dg ON ROUND(cs.display_difficulty) = dg.difficulty
WHERE cs.display_difficulty IS NOT NULL AND c.is_listed=1 AND c.layout_id=10
"""

# Query information about placements (and their mirrors)
placements_query = """
SELECT
    p.id AS placement_id,
    h.x,
    h.y,
    p.default_placement_role_id AS default_role_id,
    p.set_id AS set_id,
    s.name AS set_name,
    p_mirror.id AS mirror_placement_id
FROM placements p
JOIN holes h ON p.hole_id = h.id
JOIN sets s ON p.set_id = s.id
LEFT JOIN holes h_mirror ON h.mirrored_hole_id = h_mirror.id
LEFT JOIN placements p_mirror ON p_mirror.hole_id = h_mirror.id AND p_mirror.layout_id = p.layout_id
WHERE p.layout_id = 10
"""

# Load it into a DataFrame
df_climbs = pd.read_sql_query(climbs_query, conn)
df_placements = pd.read_sql_query(placements_query, conn)

# Save placements csv in data (for other things later on)
df_placements.to_csv('../data/placements.csv')

We've added a column for the mirror of a hold. Let's take a look at `df_placements`.

In [ ]:
display(df_placements)

In [ ]:
# Role definitions
ROLE_DEFINITIONS = {
    'start': 5,
    'middle': 6,
    'finish': 7,
    'foot': 8
}

HAND_ROLES = ['start', 'middle', 'finish']
FOOT_ROLES = ['foot']
ROLE_TYPES = ['start', 'middle', 'finish', 'hand', 'foot']

MATERIAL_PALETTE = {'Wood': '#8B4513', 'Plastic': '#4169E1'}

def get_role_type(role_id):
    """Map role_id to role_type string."""
    for role_type, rid in ROLE_DEFINITIONS.items():
        if role_id == rid:
            return role_type
    return 'unknown'

In [ ]:
# Placement Data
# Build placement_coordinates dict
placement_coordinates = {
    row['placement_id']: (row['x'], row['y'])
    for _, row in df_placements.iterrows()
}

# Build mirror mapping
placement_to_mirror = {
    row['placement_id']: int(row['mirror_placement_id'])
    for _, row in df_placements.iterrows()
    if pd.notna(row['mirror_placement_id'])
}

In [ ]:
get_role_type(7)

In [ ]:
## Boundary conditions
x_min, x_max = -68, 68
y_min, y_max = 0, 144

# Hold Usage DataFrame

In [ ]:
"""
==================================
Hold Usage DataFrame
==================================

Explodes climb frames into individual hold usages.
"""

records = []

for _, row in df_climbs.iterrows():
    frames = row['frames']
    if not isinstance(frames, str):
        continue
    
    matches = re.findall(r'p(\d+)r(\d+)', frames)
    
    for p_str, r_str in matches:
        role_type = get_role_type(int(r_str))
        records.append({
            'placement_id': int(p_str),
            'role_id': int(r_str),
            'role_type': role_type,
            'is_hand': role_type in HAND_ROLES,
            'is_foot': role_type in FOOT_ROLES,
            'difficulty': row['display_difficulty'],
            'angle': row['angle'],
            'climb_uuid': row['uuid']
        })

df_hold_usage = pd.DataFrame(records)

print(f"Built hold usage DataFrame: {len(df_hold_usage):,} records")
print(f"Unique placements: {df_hold_usage['placement_id'].nunique():,}")
print(f"Unique angles: {sorted(df_hold_usage['angle'].unique())}")

print("\nRecords by role type:")
display(df_hold_usage['role_type'].value_counts().to_frame('count'))

print(f"\nHand usages: {df_hold_usage['is_hand'].sum():,}")
print(f"Foot usages: {df_hold_usage['is_foot'].sum():,}")

# Difficulty Score

## Bayesian Smoothing of Hold Difficulty

Raw hold difficulty estimates can be unstable for rarely used holds. To reduce
noise, we apply Bayesian smoothing, shrinking hold-level averages toward the
global mean difficulty. Frequently used holds remain close to their empirical
means, while sparse holds are pulled more strongly toward the overall average.


In [ ]:
"""
==================================
Bayesian Smoothing
==================================
"""

SMOOTHING_M = 20

def bayesian_smooth(mean_col, count_col, global_mean, m=SMOOTHING_M):
    """
    Bayesian smoothing toward the global mean.
    """
    return (count_col * mean_col + m * global_mean) / (count_col + m)

GLOBAL_DIFFICULTY_MEAN = df_hold_usage['difficulty'].mean()
print(f"Global difficulty mean: {GLOBAL_DIFFICULTY_MEAN:.3f}")


## Raw Difficulty Score

In [ ]:

"""
==================================
Raw difficulty score (averged & smoothed)
==================================


Average difficulty of all climbs that use this hold, plus a Bayesian-smoothed
version that is more stable for low-usage holds.
"""

raw_scores = df_hold_usage.groupby('placement_id').agg(
    raw_difficulty=('difficulty', 'mean'),
    usage_count=('climb_uuid', 'count'),
    climbs_count=('climb_uuid', 'nunique')
)

raw_scores['raw_difficulty_smoothed'] = bayesian_smooth(
    raw_scores['raw_difficulty'],
    raw_scores['usage_count'],
    GLOBAL_DIFFICULTY_MEAN
)

raw_scores = raw_scores.round(2)

print("### Top 10 Hardest Holds (Raw)\n")
display(raw_scores.sort_values('raw_difficulty', ascending=False).head(10))

print("\n### Top 10 Easiest Holds (Raw)\n")
display(raw_scores.sort_values('raw_difficulty', ascending=True).head(10))

print("\n### Example of Raw vs Smoothed Difficulty\n")
display(raw_scores[['raw_difficulty', 'raw_difficulty_smoothed', 'usage_count']].head(10))


## Per-Angle Difficulty Score

In [ ]:
"""
==================================
Per-Angle Difficulty Score
==================================

Computes difficulty score per angle, then aggregates with weighting.
Uses Bayesian-smoothed per-angle difficulty throughout.
"""

# Calculate per-angle scores
angle_scores = df_hold_usage.groupby(['placement_id', 'angle']).agg(
    avg_difficulty=('difficulty', 'mean'),
    usage_count=('climb_uuid', 'count')
).reset_index()

# Apply Bayesian smoothing
angle_scores['avg_difficulty_smoothed'] = bayesian_smooth(
    angle_scores['avg_difficulty'],
    angle_scores['usage_count'],
    GLOBAL_DIFFICULTY_MEAN
)

# Pivot to see angles side-by-side
angle_pivot = angle_scores.pivot_table(
    index='placement_id',
    columns='angle',
    values='avg_difficulty_smoothed',
    aggfunc='mean'
)
angle_pivot.columns = [f'diff_{int(col)}deg' for col in angle_pivot.columns]

# Calculate weighted average using the smoothed per-angle values
weighted_scores = []

for pid in angle_scores['placement_id'].unique():
    df_pid = angle_scores[angle_scores['placement_id'] == pid].copy()

    total_count = df_pid['usage_count'].sum()
    weighted_diff = (
        df_pid['avg_difficulty_smoothed'] * df_pid['usage_count']
    ).sum() / total_count

    weighted_scores.append({
        'placement_id': pid,
        'angle_weighted_difficulty': weighted_diff,
        'angles_used': len(df_pid),
        'min_angle': int(df_pid['angle'].min()),
        'max_angle': int(df_pid['angle'].max()),
        'angle_range': int(df_pid['angle'].max() - df_pid['angle'].min())
    })

df_angle_scores = pd.DataFrame(weighted_scores).set_index('placement_id')

print("### Per-Angle Difficulty Analysis (Sample)\n")
display(angle_pivot.join(df_angle_scores).head(15))

print(f"\nAngles used per hold:")
print(df_angle_scores['angles_used'].describe())


## Per-Role Difficulty Score

In [ ]:
"""
==================================
Per-Role Difficulty Score
==================================

Individual roles (start, middle, finish, foot) AND aggregate (hand).
All exported difficulty values are Bayesian-smoothed.
"""

# Individual role scores
role_scores = df_hold_usage.groupby(['placement_id', 'role_type']).agg(
    avg_difficulty=('difficulty', 'mean'),
    usage_count=('climb_uuid', 'count')
).reset_index()

# Apply Bayesian smoothing
role_scores['avg_difficulty_smoothed'] = bayesian_smooth(
    role_scores['avg_difficulty'],
    role_scores['usage_count'],
    GLOBAL_DIFFICULTY_MEAN
)

# Pivot for individual roles
role_pivot = role_scores.pivot_table(
    index='placement_id',
    columns='role_type',
    values='avg_difficulty_smoothed',
    aggfunc='mean'
)
role_pivot.columns = [f'diff_as_{col}' for col in role_pivot.columns]

# Usage counts per individual role
role_counts = role_scores.pivot_table(
    index='placement_id',
    columns='role_type',
    values='usage_count',
    aggfunc='sum',
    fill_value=0
)
role_counts.columns = [f'uses_as_{col}' for col in role_counts.columns]

# Aggregate hand difficulty
hand_usage = df_hold_usage[df_hold_usage['is_hand']].groupby('placement_id').agg(
    diff_as_hand_raw=('difficulty', 'mean'),
    uses_as_hand=('climb_uuid', 'count')
)

hand_usage['diff_as_hand'] = bayesian_smooth(
    hand_usage['diff_as_hand_raw'],
    hand_usage['uses_as_hand'],
    GLOBAL_DIFFICULTY_MEAN
)

hand_usage = hand_usage[['diff_as_hand', 'uses_as_hand']]

# Combine role tables
df_role_analysis = role_pivot.join(role_counts).join(hand_usage).round(2)

cols_order = [
    'diff_as_start', 'uses_as_start',
    'diff_as_middle', 'uses_as_middle',
    'diff_as_finish', 'uses_as_finish',
    'diff_as_hand', 'uses_as_hand',
    'diff_as_foot', 'uses_as_foot'
]
cols_order = [c for c in cols_order if c in df_role_analysis.columns]
df_role_analysis = df_role_analysis[cols_order]

print("### Role-Specific Difficulty Scores (Sample)\n")
display(df_role_analysis.head(15))

print("\n### Holds Used as Both Hand and Foot\n")
dual_use = df_role_analysis[
    df_role_analysis['diff_as_hand'].notna() &
    df_role_analysis['diff_as_foot'].notna()
].copy()

if len(dual_use) > 0:
    dual_use['hand_minus_foot'] = dual_use['diff_as_hand'] - dual_use['diff_as_foot']
    display(
        dual_use[['diff_as_hand', 'diff_as_foot', 'hand_minus_foot']]
        .sort_values('hand_minus_foot', ascending=False)
        .head(15)
    )


## Per-Role Per-Angle Difficulty Score

In [ ]:
"""
==================================
Per-Role Per-Angle Difficulty Score
==================================


Granular scores: placement_id × role_type × angle
Includes both individual roles AND aggregate hand.
All downstream tables use the smoothed difficulty values.
"""

# Individual roles per angle
role_angle_scores = df_hold_usage.groupby(['placement_id', 'role_type', 'angle']).agg(
    avg_difficulty=('difficulty', 'mean'),
    usage_count=('climb_uuid', 'count')
).reset_index()

role_angle_scores['avg_difficulty_smoothed'] = bayesian_smooth(
    role_angle_scores['avg_difficulty'],
    role_angle_scores['usage_count'],
    GLOBAL_DIFFICULTY_MEAN
)

# Aggregate hand per angle
hand_angle_scores = df_hold_usage[df_hold_usage['is_hand']].groupby(['placement_id', 'angle']).agg(
    avg_difficulty=('difficulty', 'mean'),
    usage_count=('climb_uuid', 'count')
).reset_index()

hand_angle_scores['avg_difficulty_smoothed'] = bayesian_smooth(
    hand_angle_scores['avg_difficulty'],
    hand_angle_scores['usage_count'],
    GLOBAL_DIFFICULTY_MEAN
)
hand_angle_scores['role_type'] = 'hand'

# Combine all
df_role_angle = pd.concat([role_angle_scores, hand_angle_scores], ignore_index=True)

print(f"Total role-angle records: {len(df_role_angle):,}")
print("\nBreakdown by role_type:")
display(df_role_angle.groupby('role_type').size().to_frame('count'))

print("\n### Per-Role Per-Angle Difficulty Scores (Sample)\n")
display(df_role_angle.head(20))


## Creating Tables

In [ ]:
"""
==================================
Role-Specific Tables
==================================

Tables for: start, middle, finish, hand, foot
Each with per-angle columns and overall average.
Uses Bayesian-smoothed role-angle difficulty values.
"""

angles = sorted(df_hold_usage['angle'].unique())
role_tables = {}

for role in ROLE_TYPES:
    df_role = df_role_angle[df_role_angle['role_type'] == role].copy()

    if df_role.empty:
        print(f"No data for role: {role}")
        continue

    pivot = df_role.pivot_table(
        index='placement_id',
        columns='angle',
        values='avg_difficulty_smoothed',
        aggfunc='mean'
    )
    pivot.columns = [f'{role}_diff_{int(col)}deg' for col in pivot.columns]
    pivot[f'{role}_overall_avg'] = pivot.mean(axis=1).round(2)

    usage_pivot = df_role.pivot_table(
        index='placement_id',
        columns='angle',
        values='usage_count',
        aggfunc='sum',
        fill_value=0
    )
    usage_pivot.columns = [f'{role}_uses_{int(col)}deg' for col in usage_pivot.columns]
    pivot[f'{role}_total_uses'] = usage_pivot.sum(axis=1).astype(int)

    role_tables[role] = pivot.join(usage_pivot)

    print(f"\n### {role.upper()} Difficulty by Angle\n")
    display(role_tables[role].head(8))


In [ ]:
"""
==================================
Combined Table for Modelling
==================================

Build a single placement-level table used downstream in feature
engineering. The smoothed overall difficulty is exposed under the simple
name `overall_difficulty`, while the raw version is retained as
`overall_difficulty_raw` for reference.
"""

# Start with placement info
df_model_features = df_placements[['placement_id', 'x', 'y', 'set_name', 'default_role_id']].copy()
df_model_features = df_model_features.set_index('placement_id')
df_model_features = df_model_features.rename(columns={
    'set_name': 'material',
    'default_role_id': 'default_role'
})

# Add raw + smoothed overall scores
df_model_features = df_model_features.join(
    raw_scores[['raw_difficulty', 'raw_difficulty_smoothed', 'usage_count', 'climbs_count']],
    how='left'
)

# Add angle scores
df_model_features = df_model_features.join(
    df_angle_scores[['angle_weighted_difficulty', 'angles_used', 'min_angle', 'max_angle', 'angle_range']],
    how='left'
)

# Add per-role tables
for role in ROLE_TYPES:
    if role in role_tables:
        df_model_features = df_model_features.join(role_tables[role], how='left')

# Add aggregate hand / foot scores if missing
extra_role_cols = [c for c in ['diff_as_hand', 'uses_as_hand', 'diff_as_foot', 'uses_as_foot'] if c in df_role_analysis.columns]
missing_extra_cols = [c for c in extra_role_cols if c not in df_model_features.columns]
if missing_extra_cols:
    df_model_features = df_model_features.join(df_role_analysis[missing_extra_cols], how='left')

# Rename for clarity
df_model_features = df_model_features.rename(columns={
    'raw_difficulty': 'overall_difficulty_raw',
    'raw_difficulty_smoothed': 'overall_difficulty'
})

print("### Combined Model Features Table (Before Mirror)\n")
display(df_model_features.head(10))
print(f"\nShape: {df_model_features.shape}")


## Taking the Mirror Score into Account

In [ ]:
"""
==================================
Mirror average function
==================================

Mirror averaging is a simple way to stabilize difficulty estimates under
left-right board symmetry. For each mirror pair:
- if both holds have a value, we average them
- if only one side has a value, we copy it to the missing mirror hold
- metadata and usage counts are left unchanged
"""

def average_with_mirror(df, columns, placement_to_mirror):
    df_result = df.copy()
    processed = set()

    for placement_id in df_result.index:
        if placement_id in processed:
            continue

        mirror_id = placement_to_mirror.get(placement_id)

        if mirror_id and mirror_id in df_result.index:
            for col in columns:
                if col not in df_result.columns:
                    continue

                val1 = df_result.loc[placement_id, col]
                val2 = df_result.loc[mirror_id, col]

                if pd.notna(val1) and pd.notna(val2):
                    avg_val = (val1 + val2) / 2
                    df_result.loc[placement_id, col] = avg_val
                    df_result.loc[mirror_id, col] = avg_val
                elif pd.isna(val1) and pd.notna(val2):
                    df_result.loc[placement_id, col] = val2
                elif pd.notna(val1) and pd.isna(val2):
                    df_result.loc[mirror_id, col] = val1

            processed.add(mirror_id)

        processed.add(placement_id)

    return df_result


In [ ]:
"""
==================================
Apply mirror to all difficulty coluns
==================================

Averages mirror pairs for:
- overall difficulty
- angle-weighted difficulty
- per-role overall averages
- per-role per-angle difficulties
"""

overall_cols = [c for c in ['overall_difficulty', 'angle_weighted_difficulty'] if c in df_model_features.columns]
role_avg_cols = [c for c in df_model_features.columns if c.endswith('_overall_avg')]
angle_diff_cols = [c for c in df_model_features.columns if '_diff_' in c and c.endswith('deg')]

all_difficulty_cols = sorted(set(overall_cols + role_avg_cols + angle_diff_cols))

missing_before = df_model_features[all_difficulty_cols].isna().sum().sum()
print(f"Missing values before mirror: {missing_before}")
print(f"Columns affected: {len(all_difficulty_cols)}")

df_model_features = average_with_mirror(df_model_features, all_difficulty_cols, placement_to_mirror)

missing_after = df_model_features[all_difficulty_cols].isna().sum().sum()
print(f"Missing values after mirror: {missing_after}")
print(f"Reduced by: {missing_before - missing_after}")


In [ ]:
"""
==================================
Rebuild detailed role-angle table from model features
==================================

Rebuild df_role_angle from the mirror-filled placement-level table so
saved exports and later visualizations reflect the final mirrored values.
"""

records = []

angle_cols = [c for c in df_model_features.columns if '_diff_' in c and c.endswith('deg')]

for col in angle_cols:
    parts = col.split('_')
    role = parts[0]
    angle = int(parts[2].replace('deg', ''))

    for placement_id, val in df_model_features[col].dropna().items():
        records.append({
            'placement_id': placement_id,
            'role_type': role,
            'angle': angle,
            'avg_difficulty_smoothed': val,
            'usage_count': 0
        })

df_role_angle = pd.DataFrame(records)

print(f"Rebuilt role-angle table: {len(df_role_angle)} records")

print("\n### Verify Hand @ 50° Coverage\n")
hand_50 = df_role_angle[(df_role_angle['role_type'] == 'hand') & (df_role_angle['angle'] == 50)]
print(f"Records for Hand @ 50°: {len(hand_50)}")

if placement_to_mirror:
    sample_pid = list(placement_to_mirror.keys())[0]
    sample_mirror = placement_to_mirror[sample_pid]

    print(f"\nSample mirror pair {sample_pid} <-> {sample_mirror}:")
    for role in ['hand', 'foot']:
        for angle in [40, 50]:
            check = df_role_angle[
                (df_role_angle['role_type'] == role) &
                (df_role_angle['angle'] == angle) &
                (df_role_angle['placement_id'].isin([sample_pid, sample_mirror]))
            ]
            if len(check) == 2:
                vals = check['avg_difficulty_smoothed'].values
                print(f"  {role} @ {angle}°: {vals[0]:.2f} <-> {vals[1]:.2f}")


In [ ]:
"""
==================================
Verify mirror symmetry
==================================
"""

print("### Mirror Pair Verification\n")

sample_pairs = list(placement_to_mirror.items())[:5]

for pid, mirror_pid in sample_pairs:
    if pid not in df_model_features.index or mirror_pid not in df_model_features.index:
        continue
    
    print(f"Placement {pid} ↔ {mirror_pid}:")
    
    for col in ['overall_difficulty', 'hand_overall_avg', 'foot_overall_avg']:
        if col in df_model_features.columns:
            val1 = df_model_features.loc[pid, col]
            val2 = df_model_features.loc[mirror_pid, col]
            if pd.notna(val1) and pd.notna(val2):
                match = "okay" if abs(val1 - val2) < 0.01 else "x"
                print(f"  {col}: {val1:.2f} ↔ {val2:.2f} {match}")
    print()

# Count matching pairs
matching_pairs = 0
total_pairs = 0

for pid, mirror_pid in placement_to_mirror.items():
    if pid in df_model_features.index and mirror_pid in df_model_features.index:
        total_pairs += 1
        val1 = df_model_features.loc[pid, 'overall_difficulty']
        val2 = df_model_features.loc[mirror_pid, 'overall_difficulty']
        if pd.notna(val1) and pd.notna(val2) and abs(val1 - val2) < 0.01:
            matching_pairs += 1

print(f"Mirror pairs with matching values: {matching_pairs}/{total_pairs}")

In [ ]:
"""
==================================
Mirror Coverage Summary
==================================
"""

print("### Difficulty Column Coverage (After Mirror)\n")

scenarios = [
    ("Overall (all usages)", "overall_difficulty"),
    ("Angle-weighted", "angle_weighted_difficulty"),
    ("Hand (all angles)", "hand_overall_avg"),
    ("Foot (all angles)", "foot_overall_avg"),
    ("Hand @ 40°", "hand_diff_40deg"),
    ("Hand @ 50°", "hand_diff_50deg"),
    ("Foot @ 40°", "foot_diff_40deg"),
    ("Start @ 40°", "start_diff_40deg"),
    ("Finish @ 40°", "finish_diff_40deg"),
]

for name, col in scenarios:
    if col in df_model_features.columns:
        non_null = df_model_features[col].notna().sum()
        total = len(df_model_features)
        pct = non_null / total * 100
        print(f"{name:25s}: {non_null:3d}/{total} ({pct:5.1f}%)")

# Visualization

In [ ]:
"""
==================================
Visualization: difficulty heatmaps
==================================
"""

os.makedirs('../images/03_hold_difficulty', exist_ok=True)

def plot_difficulty_heatmap(score_column='overall_difficulty', title_suffix="", save=True):
    """Plot hold difficulty scores on the board."""
    
    fig, ax = plt.subplots(figsize=(16, 14))
    ax.imshow(board_img, extent=[x_min, x_max, y_min, y_max], aspect='auto')
    
    df_plot = df_model_features[df_model_features['x'].notna()].copy()
    
    if score_column not in df_plot.columns:
        print(f"Column '{score_column}' not found")
        plt.close()
        return
    
    df_plot = df_plot[df_plot[score_column].notna()]
    
    if df_plot.empty:
        print(f"No data for '{score_column}'")
        plt.close()
        return
    
    max_usage = df_plot['usage_count'].max()
    size_scale = 20 + 150 * (df_plot['usage_count'] / max_usage)
    
    scatter = ax.scatter(
        df_plot['x'],
        df_plot['y'],
        c=df_plot[score_column],
        s=size_scale,
        cmap='coolwarm',
        alpha=0.85,
        edgecolors='black',
        linewidths=0.5
    )
    
    ax.set_xlabel('X Position (inches)', fontsize=12)
    ax.set_ylabel('Y Position (inches)', fontsize=12)
    ax.set_title(f'Hold Difficulty: {score_column} {title_suffix}', fontsize=14)
    
    cbar = plt.colorbar(scatter, ax=ax, shrink=0.5)
    cbar.set_label('Difficulty')
    
    plt.tight_layout()
    
    if save:
        safe_name = score_column.replace('/', '_')
        plt.savefig(f'../images/03_hold_difficulty/difficulty_heatmap_{safe_name}.png', dpi=150, bbox_inches='tight')
    
    plt.show()


# Plot main scores
plot_difficulty_heatmap('overall_difficulty', "(Raw Average)")
plot_difficulty_heatmap('angle_weighted_difficulty', "(Angle-Weighted)")

# Plot role scores
plot_difficulty_heatmap('hand_overall_avg', "(Hand)")
plot_difficulty_heatmap('foot_overall_avg', "(Foot)")

In [ ]:
"""
==================================
Visualization: per-role per-angle heatmaps
==================================
"""

def plot_role_angle_heatmap(role_type='hand', angle=40):
    """Plot difficulty scores for a specific role and angle."""
        
    df_role = df_role_angle[
        (df_role_angle['role_type'] == role_type) & 
        (df_role_angle['angle'] == angle)
    ].copy()
    
    if df_role.empty:
        print(f"No data for {role_type} at {angle}°")
        return
    
    df_role['x'] = df_role['placement_id'].map(lambda p: placement_coordinates.get(p, (None, None))[0])
    df_role['y'] = df_role['placement_id'].map(lambda p: placement_coordinates.get(p, (None, None))[1])
    df_role = df_role.dropna(subset=['x', 'y'])
    
    fig, ax = plt.subplots(figsize=(16, 14))
    ax.imshow(board_img, extent=[x_min, x_max, y_min, y_max], aspect='auto')
    
    scatter = ax.scatter(
        df_role['x'],
        df_role['y'],
        c=df_role['avg_difficulty_smoothed'],
        s=100,
        cmap='coolwarm',
        alpha=0.85,
        edgecolors='black',
        linewidths=0.5
    )
    
    ax.set_xlabel('X Position (inches)', fontsize=12)
    ax.set_ylabel('Y Position (inches)', fontsize=12)
    ax.set_title(f'{role_type.capitalize()} Hold Difficulty at {angle}°', fontsize=14)
    
    cbar = plt.colorbar(scatter, ax=ax, shrink=0.5)
    cbar.set_label('Difficulty')
    
    plt.tight_layout()
    plt.savefig(f'../images/03_hold_difficulty/difficulty_{role_type}_{angle}deg.png', dpi=150, bbox_inches='tight')
    plt.show()


# Plot for common angles
common_angles = [30, 40, 45, 50]

for role in ['hand', 'foot']:
    for angle in common_angles:
        print(f"\n{role.capitalize()} at {angle}°:")
        plot_role_angle_heatmap(role, angle)

# Conclusion

In [ ]:
"""
==================================
Summary Statistics
==================================
"""

# Material comparison
print("### Difficulty by Material\n")
material_diff = df_model_features.groupby('material').agg(
    count=('overall_difficulty', 'count'),
    avg_difficulty=('overall_difficulty', 'mean'),
    median_difficulty=('overall_difficulty', 'median'),
    avg_hand=('hand_overall_avg', 'mean'),
    avg_foot=('foot_overall_avg', 'mean'),
    avg_usage=('usage_count', 'mean')
).round(2)

display(material_diff)

# Default role comparison
print("\n### Difficulty by Default Role\n")
role_diff = df_model_features.groupby('default_role').agg(
    count=('overall_difficulty', 'count'),
    avg_difficulty=('overall_difficulty', 'mean'),
    avg_hand=('hand_overall_avg', 'mean'),
    avg_foot=('foot_overall_avg', 'mean'),
    avg_usage=('usage_count', 'mean')
).round(2)

display(role_diff)

# Correlation
if 'hand_overall_avg' in df_model_features.columns and 'foot_overall_avg' in df_model_features.columns:
    valid = df_model_features.dropna(subset=['hand_overall_avg', 'foot_overall_avg'])
    if len(valid) > 0:
        corr = valid['hand_overall_avg'].corr(valid['foot_overall_avg'])
        print(f"\nCorrelation (hand vs foot difficulty): {corr:.3f}")
        print(f"(based on {len(valid)} holds used as both)")

In [ ]:
"""
==================================
Save to  files
==================================
"""

import os
os.makedirs('../data/03_hold_difficulty', exist_ok=True)

# Main features table
df_model_features.to_csv('../data/03_hold_difficulty/hold_difficulty_scores.csv')

# Full pivot for modeling
pivot_value_col = 'avg_difficulty_smoothed' if 'avg_difficulty_smoothed' in df_role_angle.columns else 'avg_difficulty'

pivot_full = df_role_angle.pivot_table(
    index='placement_id',
    columns=['role_type', 'angle'],
    values=pivot_value_col,
    aggfunc='mean'
)
pivot_full.columns = [f'diff_{role}_{int(angle)}deg' for role, angle in pivot_full.columns]
pivot_full.to_csv('../data/03_hold_difficulty/hold_role_angle_difficulty_scores.csv')

# Per-role tables
for role in ROLE_TYPES:
    if role in role_tables:
        role_tables[role].to_csv(f'../data/03_hold_difficulty/hold_{role}_difficulty_by_angle.csv')

# Detailed records
df_role_angle.to_csv('../data/03_hold_difficulty/hold_role_angle_detailed.csv', index=False)

print("Saved files to ../data/03_hold_difficulty/:")
print("  - hold_difficulty_scores.csv (main table)")
print("  - hold_role_angle_difficulty_scores.csv (full pivot)")
for role in ROLE_TYPES:
    if role in role_tables:
        print(f"  - hold_{role}_difficulty_by_angle.csv")
print("  - hold_role_angle_detailed.csv (detailed records)")



## Tables produced:

1. `df_model_features` - Main feature table for downstream modeling
   - One row per `placement_id`
   - Includes metadata, overall scores, angle-level summaries, and role-specific scores
   - `overall_difficulty` is the Bayesian-smoothed overall score
   - `overall_difficulty_raw` is retained only as a reference column

2. `df_role_angle` - Detailed records for visualization / export
   - One row per (`placement_id`, `role_type`, `angle`) combination
   - Rebuilt after mirror-averaging so plots and exports reflect the final mirrored values

3. `role_tables[role]` - Per-role tables
   - start, middle, finish, hand, foot
   - each with per-angle columns, overall averages, and usage counts

## Mirror logic:
- difficulty-like columns are mirrored across symmetric hold pairs
- if both mirror holds have values, their scores are averaged
- if only one side has a value, that value is copied to the missing mirror side
- usage counts and metadata are left unchanged